In [ ]:
%matplotlib inline

In [ ]:
import os
import pickle

import pandas as pd
from rdkit import Chem, rdBase
from tqdm import tqdm

from src import morgan_tokenize, smiles_tokenize

rdBase.DisableLog('rdApp.warning')

In [ ]:
csv_data = os.path.abspath(
    r"D:\Datasets\Corpus_ZINC20\ZINC20_3D_Standard_In-Stock_Ref+Mid_-2~2_smi_WGET_Flat\ZINC20_raw_10116577.csv")
col = "canonical_smiles"
special_tokens = {"<pad>": 0, "<unk>": 1, "<cls>": 2, "<sep>": 3, "<mask>": 4}

$🔧 \textcolor{cyan}{\textbf{\textit{Build vocab of SMILES}}}$

In [ ]:
# Initialize counters for data quality tracking
total, valid = 0, 0

# Initialize vocabulary
generated_vocab = special_tokens.copy()
vocab_size = len(generated_vocab)

# Load raw chemical data
df = pd.read_csv(csv_data, low_memory=False)

print("Building Vocab...")
for smi in tqdm(df[col]):
    mol = Chem.MolFromSmiles(smi)

    if mol:
        tokens = smiles_tokenize(smi)
        for t in tokens:
            if t and t not in generated_vocab.keys():
                generated_vocab[t] = vocab_size
                vocab_size += 1
        valid += 1
    total += 1

# Serialize the mapping for consistency during training and inference
with open('../vocab_smiles.pickle', 'wb') as f:
    pickle.dump(generated_vocab, f)

# Summary statistics for dataset audit
print(f"Total SMILES processed: {total}")
print(f"Valid SMILES identified: {valid}")
print(f"Final Vocab Size: {vocab_size}")

$🔧 \textcolor{cyan}{\textbf{\textit{Build vocab of MorganFP}}}$

In [ ]:
# Initialize counters for data quality tracking
total, valid = 0, 0

# Initialize vocabulary
generated_vocab = special_tokens.copy()
vocab_size = len(generated_vocab)

# Load raw chemical data
df = pd.read_csv(csv_data, low_memory=False)

print("Building Vocab...")
for smi in tqdm(df[col]):
    mol = Chem.MolFromSmiles(smi)

    if mol:
        tokens_r0 = morgan_tokenize(smi, 0, True)
        tokens_r1 = morgan_tokenize(smi, 1, True)
        if 2 * len(tokens_r0) == len(tokens_r1):
            for token in tokens_r1:
                if token and token not in generated_vocab.keys():
                    generated_vocab[token] = vocab_size
                    vocab_size += 1
            valid += 1
    total += 1

# Serialize the mapping for consistency during training and inference
with open('../vocab_morgan.pickle', 'wb') as f:
    pickle.dump(generated_vocab, f)

# Summary statistics for dataset audit
print(f"Total SMILES processed: {total}")
print(f"Valid SMILES identified: {valid}")
print(f"Final Vocab Size: {vocab_size}")